# Generating continent-scale Land Cover 2.0 imagery with Australian bathymetry

This notebook uses DEA [Land Cover 2.0](https://knowledge.dea.ga.gov.au/data/product/dea-land-cover-landsat/) and [The AusBathyTopo 250m (Australia) 2023 Grid](https://knowledge.dea.ga.gov.au/data/external-data/ga-ausbathytopo-250m/) to create yearly images of the Australian continent. 

The notebook also uses a custom projection, based on Australian Albers (EPSG:3577) to allow for the northern extens of Australia to be included. The extent spatial extent of EPSG:3577 is: `[112.85, -43.7, 153.69, -9.86]`. 

The custom projection uses the same ellipsoid, but removes the spatial extents.


In [ ]:
%pip install rioxarray -q

In [ ]:
%matplotlib inline

import gc
import sys
import rasterio
import geopandas as gpd
import datacube
import numpy as np
import xarray as xr
import rioxarray as rxr
import cartopy
from rasterio.features import rasterize
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from datacube.utils.cog import write_cog
from matplotlib import colors as mcolours

sys.path.insert(1, "../Tools/")
from dea_tools.plotting import display_map
from dea_tools.landcover import plot_land_cover
from dea_tools.dask import create_local_dask_cluster
from dea_tools.dask import create_dask_gateway_cluster

In [ ]:
client = create_local_dask_cluster(return_client=True)

In [ ]:
dc = datacube.Datacube(app='lc_images_for_animation')

## Add the Land Cover colour schemes

- While there are existing animation and plotting tools in `dea_tools` some customisation options are not currently in them. So, we copy the Level 3 and Level 4 colour schemes into this notebook to use them in our custom image function.

In [ ]:
LEVEL3_COLOUR_SCHEME = {
    111: (172, 188, 45, 255, "Cultivated terrestrial vegetation"),
    112: (14, 121, 18, 255, "Natural terrestrial vegetation"),
    124: (30, 191, 121, 255, "Natural aquatic vegetation"),
    215: (218, 92, 105, 255, "Artificial surface"),
    216: (243, 171, 105, 255, "Natural bare surface"),
    220: (77, 159, 220, 255, "Water"),
    255: (255, 255, 255, 255, "No Data"),
}

In [ ]:
LEVEL4_COLOUR_SCHEME = {
               1: (151, 187, 26, 255, 'Cultivated Terrestrial\n Vegetated:'),
               2: (151, 187, 26, 255, 'Cultivated Terrestrial\n Vegetated: Woody'),
               3: (209, 224, 51, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous'),
               4: (197, 168, 71, 255, 'Cultivated Terrestrial\n Vegetated: Closed\n (> 65 %)'),
               5: (205, 181, 75, 255, 'Cultivated Terrestrial\n Vegetated: Open\n (40 to 65 %)'),
               6: (213, 193, 79, 255, 'Cultivated Terrestrial\n Vegetated: Open\n (15 to 40 %)'),
               7: (228, 210, 108, 255, 'Cultivated Terrestrial\n Vegetated: Sparse\n (4 to 15 %)'),
               8: (242, 227, 138, 255, 'Cultivated Terrestrial\n Vegetated: Scattered\n (1 to 4 %)'),
               9: (197, 168, 71, 255, 'Cultivated Terrestrial\n Vegetated: Woody Closed\n (> 65 %)'),
               10: (205, 181, 75, 255, 'Cultivated Terrestrial\n Vegetated: Woody Open\n (40 to 65 %)'),
               11: (213, 193, 79, 255, 'Cultivated Terrestrial\n Vegetated: Woody Open\n (15 to 40 %)'),
               12: (228, 210, 108, 255, 'Cultivated Terrestrial\n Vegetated: Woody Sparse\n (4 to 15 %)'),
               13: (242, 227, 138, 255, 'Cultivated Terrestrial\n Vegetated: Woody Scattered\n (1 to 4 %)'),
               14: (228, 224, 52, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Closed\n (> 65 %)'),
               15: (235, 232, 84, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Open\n (40 to 65 %)'),
               16: (242, 240, 127, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Open\n (15 to 40 %)'),
               17: (249, 247, 174, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Sparse\n (4 to 15 %)'),
               18: (255, 254, 222, 255, 'Cultivated Terrestrial\n Vegetated: Herbaceous Scattered\n (1 to 4 %)'),
               19: (14, 121, 18, 255, 'Natural Terrestrial Vegetated:'),
               20: (26, 177, 87, 255, 'Natural Terrestrial Vegetated: Woody'),
               21: (94, 179, 31, 255, 'Natural Terrestrial Vegetated: Herbaceous'),
               22: (14, 121, 18, 255, 'Natural Terrestrial Vegetated: Closed (> 65 %)'),
               23: (45, 141, 47, 255, 'Natural Terrestrial Vegetated: Open (40 to 65 %)'),
               24: (80, 160, 82, 255, 'Natural Terrestrial Vegetated: Open (15 to 40 %)'),
               25: (117, 180, 118, 255, 'Natural Terrestrial Vegetated: Sparse (4 to 15 %)'),
               26: (154, 199, 156, 255, 'Natural Terrestrial Vegetated: Scattered (1 to 4 %)'),
               27: (14, 121, 18, 255, 'Natural Terrestrial Vegetated: Woody Closed (> 65 %)'),
               28: (45, 141, 47, 255, 'Natural Terrestrial Vegetated: Woody Open (40 to 65 %)'),
               29: (80, 160, 82, 255, 'Natural Terrestrial Vegetated: Woody Open (15 to 40 %)'),
               30: (117, 180, 118, 255, 'Natural Terrestrial Vegetated: Woody Sparse (4 to 15 %)'),
               31: (154, 199, 156, 255, 'Natural Terrestrial Vegetated: Woody Scattered (1 to 4 %)'),
               32: (119, 167, 30, 255, 'Natural Terrestrial Vegetated: Herbaceous Closed (> 65 %)'),
               33: (136, 182, 51, 255, 'Natural Terrestrial Vegetated: Herbaceous Open (40 to 65 %)'),
               34: (153, 196, 80, 255, 'Natural Terrestrial Vegetated: Herbaceous Open (15 to 40 %)'),
               35: (170, 212, 113, 255, 'Natural Terrestrial Vegetated: Herbaceous Sparse (4 to 15 %)'),
               36: (186, 226, 146, 255, 'Natural Terrestrial Vegetated: Herbaceous Scattered (1 to 4 %)'),
               37: (86, 236, 231, 255, 'Cultivated Aquatic Vegetated:'),
               38: (61, 170, 140, 255, 'Cultivated Aquatic Vegetated: Woody'),
               39: (82, 231, 172, 255, 'Cultivated Aquatic Vegetated: Herbaceous'),
               40: (43, 210, 203, 255, 'Cultivated Aquatic Vegetated: Closed (> 65 %)'),
               41: (73, 222, 216, 255, 'Cultivated Aquatic Vegetated: Open (40 to 65 %)'),
               42: (110, 233, 228, 255, 'Cultivated Aquatic Vegetated: Open (15 to 40 %)'),
               43: (149, 244, 240, 255, 'Cultivated Aquatic Vegetated: Sparse (4 to 15 %)'),
               44: (187, 255, 252, 255, 'Cultivated Aquatic Vegetated: Scattered (1 to 4 %)'),
               50: (82, 231, 196, 255, 'Cultivated Aquatic Vegetated: Herbaceous Closed (> 65 %)'),
               51: (113, 237, 208, 255, 'Cultivated Aquatic Vegetated: Herbaceous Open (40 to 65 %)'),
               52: (144, 243, 220, 255, 'Cultivated Aquatic Vegetated: Herbaceous Open (15 to 40 %)'),
               53: (175, 249, 232, 255, 'Cultivated Aquatic Vegetated: Herbaceous Sparse (4 to 15 %)'),
               54: (207, 255, 244, 255, 'Cultivated Aquatic Vegetated: Herbaceous Scattered (1 to 4 %)'),
               55: (30, 191, 121, 255, 'Natural Aquatic Vegetated:'),
               56: (18, 142, 148, 255, 'Natural Aquatic Vegetated: Woody'),
               57: (112, 234, 134, 255, 'Natural Aquatic Vegetated: Herbaceous'),
               58: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Closed (> 65 %)'),
               59: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Open (40 to 65 %)'),
               60: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Open (15 to 40 %)'),
               61: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Sparse (4 to 15 %)'),
               62: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Scattered (1 to 4 %)'),
               63: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %)'),
               64: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %) Water > 3 months (semi-) permenant'),
               65: (25, 173, 109, 255, 'Natural Aquatic Vegetated: Woody Closed (> 65 %) Water < 3 months (temporary or seasonal)'),
               66: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %)'),
               67: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %) Water > 3 months (semi-) permenant'),
               68: (53, 184, 132, 255, 'Natural Aquatic Vegetated: Woody Open (40 to 65 %) Water < 3 months (temporary or seasonal)'),
               69: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %)'),
               70: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %) Water > 3 months (semi-) permenant'),
               71: (93, 195, 155, 255, 'Natural Aquatic Vegetated: Woody Open (15 to 40 %) Water < 3 months (temporary or seasonal)'),
               72: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %)'),
               73: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %) Water > 3 months (semi-) permenant'),
               74: (135, 206, 178, 255, 'Natural Aquatic Vegetated: Woody Sparse (4 to 15 %) Water < 3 months (temporary or seasonal)'),
               75: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %)'),
               76: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %) Water > 3 months (semi-) permenant'),
               77: (176, 218, 201, 255, 'Natural Aquatic Vegetated: Woody Scattered (1 to 4 %) Water < 3 months (temporary or seasonal)'),
               78: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %)'),
               79: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %) Water > 3 months (semi-) permenant'),
               80: (39, 204, 139, 255, 'Natural Aquatic Vegetated: Herbaceous Closed (> 65 %) Water < 3 months (temporary or seasonal)'),
               81: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %)'),
               82: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %) Water > 3 months (semi-) permenant'),
               83: (66, 216, 159, 255, 'Natural Aquatic Vegetated: Herbaceous Open (40 to 65 %) Water < 3 months (temporary or seasonal)'),
               84: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %)'),
               85: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %) Water > 3 months (semi-) permenant'),
               86: (99, 227, 180, 255, 'Natural Aquatic Vegetated: Herbaceous Open (15 to 40 %) Water < 3 months (temporary or seasonal)'),
               87: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %)'),
               88: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %) Water > 3 months (semi-) permenant'),
               89: (135, 239, 201, 255, 'Natural Aquatic Vegetated: Herbaceous Sparse (4 to 15 %) Water < 3 months (temporary or seasonal)'),
               90: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %)'),
               91: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %) Water > 3 months (semi-) permenant'),
               92: (171, 250, 221, 255, 'Natural Aquatic Vegetated: Herbaceous Scattered (1 to 4 %) Water < 3 months (temporary or seasonal)'),
               93: (218, 92, 105, 255, 'Artificial Surface:'),
               94: (243, 171, 105, 255, 'Natural Surface:'),
               95: (255, 230, 140, 255, 'Natural Surface: Sparsely vegetated'),
               96: (250, 210, 110, 255, 'Natural Surface: Very sparsely vegetated'),
               97: (243, 171, 105, 255, 'Natural Surface: Bare areas, unvegetated'),
               98: (77, 159, 220, 255, 'Water:'),
               99: (77, 159, 220, 255, 'Water: (Water)'),
               100: (187, 220, 233, 255, 'Water: (Water) Tidal area'),
               101: (27, 85, 186, 255, 'Water: (Water) Perennial (> 9 months)'),
               102: (52, 121, 201, 255, 'Water: (Water) Non-perennial (7 to 9 months)'),
               103: (79, 157, 217, 255, 'Water: (Water) Non-perennial (4 to 6 months)'),
               104: (133, 202, 253, 255, 'Water: (Water) Non-perennial (1 to 3 months)'),
               255: (255, 255, 255, 255, "No Data")
               }

## Create custom projection based on Australian Albers (3577)

- The reason for doing this is that the spatial extent of EPSG:3577 results in the northern extents of some islands and the top of the continent being excluded from the cartopy generatd imagery below
- Note: this is not an issue with the underlying data being pulled from ODC, it is an issue with Cartopy visualisations

In [ ]:
aus_albers = ccrs.AlbersEqualArea(
    central_longitude=132,
    standard_parallels=[-18, -36],
    globe=cartopy.crs.Globe(ellipse='GRS80')
)

## Custom function to generate yearly images

- use of `gc.collect()` to try and minimise memory usage when generating large images from the fetched data

In [ ]:
def plot_layer(colours, data, data_2):
    minx, miny, maxx, maxy = data.rio.bounds()
    colour_arr = [np.array(value[:-1]) / 255 for value in colours.values()]
    cmap = mcolours.ListedColormap(colour_arr)
    bounds = list(colours) + [256]
    norm = mcolours.BoundaryNorm(np.array(bounds) - 0.1, cmap.N)

    fig, ax = plt.subplots(figsize=(10, 10), dpi=200, subplot_kw={'projection': aus_albers})
    data_2.plot(ax=ax, cmap='Blues_r', alpha=0.75, add_colorbar=False, transform=ccrs.epsg(3577))
    im = data.plot(ax=ax, cmap=cmap, norm=norm, add_colorbar=False, transform=ccrs.epsg(3577))
    
    ax.set_title('')
    ax.set_extent([minx, maxx, miny, maxy], crs=ccrs.epsg(3577))

    year = str(time.dt.year.values)
    ax.text(0.9, 0.9, f'{year}', transform=ax.transAxes, ha='center', va='bottom', fontsize=22, bbox=dict(facecolor='white', alpha=0.8))
    plt.savefig(f'outputs/land_cover_level4_1km_{year}.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

    # Clear out the used data
    del fig, ax, im
    gc.collect()

## Filter out data over GQA issue tiles in WA using a custom filter

In [ ]:
def filter_tiles(dataset):
    # Don't return data for region codes X and X, except if timestep is after 2005
    return (dataset.metadata.region_code not in ["x39y49", "x41y45", "x40y47"]) or (dataset.time.begin.year < 2005)

## Define the extents for the images/visualisations

In [ ]:
lat_range = (-9.86, -43.7)
lon_range = (112.85, 153.69)
time = ('2022','2023')

In [ ]:
query = {
    'time':time
}

landcover_ds = dc.load(product='ga_ls_landcover_class_cyear_3',
                 measurements='level4',
                 output_crs='EPSG:3577',
                dask_chunks={},
                resolution=(-1000, 1000),
                  dataset_predicate=filter_tiles,
                 **query)

In [ ]:
bathy_ds = dc.load(
    product='ga_ausbathytopo250m_2023',
    output_crs='EPSG:3577',
    resolution=(-1000,1000),
    dask_chunks={},
    **query
)

## Import Australian coastlines to use for masking

- DEA Land Cover is intended for use over land, not open waters. So, to improve the appearance of the final image and to remove the data over open waters, we will import the Australian coastlines shapefile and use this to make a bit mask

In [ ]:
# Get coastlines vector data to later use for masking
coastlines_shp = 'australia/cstauscd_r.shp'
coastlines_gdf = gpd.read_file(coastlines_shp).to_crs('EPSG:3577')
coastlines_gdf = coastlines_gdf.query("STATE_CODE != 0")
coastlines_gdf.crs


In [ ]:
# Convert the polygons to a mask
shapes = [(geom, 1) for geom in coastlines_gdf.geometry]
mask = rasterize(
    shapes,
    out_shape=landcover_ds.rio.shape,
    transform=landcover_ds.rio.transform(),
    fill=0,
    dtype='uint8'
)

## Apply mask to Land Cover and bathymetry data
- There is bathy data over the continent. To reduce the data being loaded and also to make sure the colour ramp is being applied as expected, we mask out data over the continent.
- Becuase the bathy data is not the primary focus here, I've also capped the depth at 5000m below sea level
- Land Cover open water data (pixels over oceans, seas) are also masked out so that the bathy data can be seen around the coastlines

In [ ]:
masked_ds = landcover_ds.where(mask)

masked_bathy = bathy_ds.where((bathy_ds < 10) & (bathy_ds >= -5000))
masked_bathy = masked_bathy.where(masked_bathy >= -5000, -5000)

- The bathymetry data is currently in a DataSet - but we need to individual Data Array inorder to plot it. So, select the array that contains the bathymetry data:

In [ ]:
masked_bathy1 = masked_bathy.to_array().isel(variable=0)

In [ ]:
data = masked_ds.level4

In [ ]:
for i in range(len(data['time'])):
    ds = data.isel(time=i).load()
    plot_layer(LEVEL4_COLOUR_SCHEME, ds, masked_bathy1)
    del ds
    gc.collect()
    print(f'{i} processed')

In [ ]:
# for time in data.time:
#     plot_layer(LEVEL4_COLOUR_SCHEME, data, masked_bathy1, time)
#     gc.collect()